# 3. Classification Models
## Predicting Price Segments (Low/Medium/High)

## 3.1 Import Libraries

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
import joblib
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully!')

## 3.2 Load Preprocessed Data

In [ ]:
# Load cleaned data
df_clean = pd.read_csv('../data_cleaned.csv')

print(f'Dataset shape: {df_clean.shape}')
print(f'Columns: {list(df_clean.columns)}')

## 3.3 Create Price Segments (Target Variable)

In [ ]:
# Define price segments based on percentiles
low_threshold = df_clean['Price'].quantile(0.33)
high_threshold = df_clean['Price'].quantile(0.67)

print(f'Price percentiles:')
print(f'  33rd percentile (Low/Medium boundary): {low_threshold:.2f} Billion VND')
print(f'  67th percentile (Medium/High boundary): {high_threshold:.2f} Billion VND')

# Create price segment column
def categorize_price(price):
    if price < low_threshold:
        return 'Low'
    elif price < high_threshold:
        return 'Medium'
    else:
        return 'High'

df_clean['Price_Segment'] = df_clean['Price'].apply(categorize_price)

# Show distribution
print(f'\nPrice Segment Distribution:')
print(df_clean['Price_Segment'].value_counts())

In [ ]:
# Visualize price segments
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
segment_counts = df_clean['Price_Segment'].value_counts()
colors = {'Low': '#2ecc71', 'Medium': '#f39c12', 'High': '#e74c3c'}
axes[0].pie(segment_counts.values, labels=segment_counts.index, 
            autopct='%1.1f%%', 
            colors=[colors[s] for s in segment_counts.index])
axes[0].set_title('Price Segment Distribution', fontsize=13, fontweight='bold')

# Bar chart
segment_counts.plot(kind='bar', ax=axes[1], color=[colors[s] for s in segment_counts.index])
axes[1].set_xlabel('Price Segment')
axes[1].set_ylabel('Count')
axes[1].set_title('Price Segment Counts', fontsize=13, fontweight='bold')
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=0)

# Add value labels
for i, v in enumerate(segment_counts.values):
    axes[1].text(i, v + 50, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../figures/11_price_segments.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.4 Prepare Features for Classification

In [ ]:
# Select features for classification
# Numerical features
numerical_features = ['Area', 'Frontage', 'Access Road', 'Floors', 'Bedrooms', 'Bathrooms']

# Categorical features for encoding
categorical_features = ['House direction', 'Balcony direction', 'Legal status', 'Furniture state', 'City']

print('Features for classification:')
print(f'  Numerical: {numerical_features}')
print(f'  Categorical: {categorical_features}')

In [ ]:
# Apply one-hot encoding to categorical features
df_model = df_clean.copy()

# Fill any remaining missing values
for col in numerical_features:
    df_model[col] = df_model[col].fillna(df_model[col].median())

for col in categorical_features:
    df_model[col] = df_model[col].fillna(df_model[col].mode()[0])

# One-hot encode categorical features
df_encoded = pd.get_dummies(df_model[categorical_features], drop_first=False)

# Combine numerical and encoded features
X = pd.concat([df_model[numerical_features], df_encoded], axis=1)
y = df_model['Price_Segment']

print(f'\nFeature matrix shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'\nNumber of features after encoding: {X.shape[1]}')

In [ ]:
# Save feature names for later use
feature_names = list(X.columns)
joblib.dump(feature_names, '../models/feature_names.pkl')
print('Feature names saved to: models/feature_names.pkl')

## 3.5 Train/Test Split

In [ ]:
# Split data: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y
)

print('=== TRAIN/TEST SPLIT ===\n')
print(f'Training set: {X_train.shape[0]:,} samples')
print(f'Test set: {X_test.shape[0]:,} samples')
print(f'\nTraining set class distribution:')
print(y_train.value_counts())
print(f'\nTest set class distribution:')
print(y_test.value_counts())

## 3.6 Train Decision Tree Classifier

In [ ]:
# Decision Tree Classifier
dt_model = DecisionTreeClassifier(
    max_depth=10,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42
)

# Train
dt_model.fit(X_train, y_train)

# Predict
y_pred_dt = dt_model.predict(X_test)

print('Decision Tree Classifier trained successfully!')

In [ ]:
# Decision Tree Evaluation
print('=== DECISION TREE RESULTS ===\n')

print('Accuracy:', accuracy_score(y_test, y_pred_dt).round(4))
print('\nPrecision (macro avg):', precision_score(y_test, y_pred_dt, average='macro').round(4))
print('Recall (macro avg):', recall_score(y_test, y_pred_dt, average='macro').round(4))
print('F1 Score (macro avg):', f1_score(y_test, y_pred_dt, average='macro').round(4))

print('\nClassification Report:')
print(classification_report(y_test, y_pred_dt, target_names=['High', 'Low', 'Medium']))

In [ ]:
# Decision Tree Confusion Matrix
fig, ax = plt.subplots(figsize=(8, 6))

cm_dt = confusion_matrix(y_test, y_pred_dt, labels=['High', 'Low', 'Medium'])
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['High', 'Low', 'Medium'],
            yticklabels=['High', 'Low', 'Medium'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Decision Tree - Confusion Matrix', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../figures/12_dt_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.7 Train Random Forest Classifier

In [ ]:
# Random Forest Classifier
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

# Train
rf_model.fit(X_train, y_train)

# Predict
y_pred_rf = rf_model.predict(X_test)

print('Random Forest Classifier trained successfully!')

In [ ]:
# Random Forest Evaluation
print('=== RANDOM FOREST RESULTS ===\n')

print('Accuracy:', accuracy_score(y_test, y_pred_rf).round(4))
print('\nPrecision (macro avg):', precision_score(y_test, y_pred_rf, average='macro').round(4))
print('Recall (macro avg):', recall_score(y_test, y_pred_rf, average='macro').round(4))
print('F1 Score (macro avg):', f1_score(y_test, y_pred_rf, average='macro').round(4))

print('\nClassification Report:')
print(classification_report(y_test, y_pred_rf, target_names=['High', 'Low', 'Medium']))

In [ ]:
# Random Forest Confusion Matrix
fig, ax = plt.subplots(figsize=(8, 6))

cm_rf = confusion_matrix(y_test, y_pred_rf, labels=['High', 'Low', 'Medium'])
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens', 
            xticklabels=['High', 'Low', 'Medium'],
            yticklabels=['High', 'Low', 'Medium'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Random Forest - Confusion Matrix', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../figures/13_rf_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.8 Model Comparison

In [ ]:
# Comparison table
metrics = {
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'Decision Tree': [
        accuracy_score(y_test, y_pred_dt),
        precision_score(y_test, y_pred_dt, average='macro'),
        recall_score(y_test, y_pred_dt, average='macro'),
        f1_score(y_test, y_pred_dt, average='macro')
    ],
    'Random Forest': [
        accuracy_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_rf, average='macro'),
        recall_score(y_test, y_pred_rf, average='macro'),
        f1_score(y_test, y_pred_rf, average='macro')
    ]
}

comparison_df = pd.DataFrame(metrics)
comparison_df['Random Forest'] = comparison_df['Random Forest'].round(4)
comparison_df['Decision Tree'] = comparison_df['Decision Tree'].round(4)

print('=== MODEL COMPARISON ===\n')
print(comparison_df.to_string(index=False))

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(comparison_df))
width = 0.35

bars1 = ax.bar(x - width/2, comparison_df['Decision Tree'], width, label='Decision Tree', color='#3498db')
bars2 = ax.bar(x + width/2, comparison_df['Random Forest'], width, label='Random Forest', color='#2ecc71')

ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(comparison_df['Metric'])
ax.legend()
ax.set_ylim(0, 1)

# Add value labels
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.3f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha='center', va='bottom', fontsize=9)

for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.3f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('../figures/14_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.9 Feature Importance (Random Forest)

In [ ]:
# Feature importance from Random Forest
feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

# Plot top 20 features
fig, ax = plt.subplots(figsize=(10, 8))

top_features = feature_importance.head(20)
colors = plt.cm.viridis(np.linspace(0, 0.8, len(top_features)))

ax.barh(range(len(top_features)), top_features['Importance'].values, color=colors)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['Feature'].values)
ax.invert_yaxis()
ax.set_xlabel('Importance')
ax.set_title('Top 20 Feature Importance (Random Forest)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../figures/15_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10 Most Important Features:')
print(feature_importance.head(10).to_string(index=False))

## 3.10 Save Trained Models

In [ ]:
# Save models
joblib.dump(dt_model, '../models/decision_tree.pkl')
joblib.dump(rf_model, '../models/random_forest.pkl')

print('=== MODELS SAVED ===\n')
print('  - models/decision_tree.pkl')
print('  - models/random_forest.pkl')
print('  - models/feature_names.pkl')

## 3.11 Prediction Function

In [ ]:
# Prediction function for new data
def predict_price_segment(area, frontage, access_road, floors, bedrooms, bathrooms,
                         legal_status, furniture_state, city):
    """
    Predict price segment for a property
    """
    # Load model and feature names
    model = joblib.load('../models/random_forest.pkl')
    feature_names = joblib.load('../models/feature_names.pkl')
    
    # Create input dataframe
    input_data = {
        'Area': [area],
        'Frontage': [frontage],
        'Access Road': [access_road],
        'Floors': [floors],
        'Bedrooms': [bedrooms],
        'Bathrooms': [bathrooms]
    }
    
    # Add encoded categorical features
    for name in feature_names:
        if name.startswith('House direction_'):
            input_data[name] = [1 if name == f'House direction_{legal_status}' else 0]
        elif name.startswith('Legal status_'):
            input_data[name] = [1 if name == f'Legal status_{legal_status}' else 0]
        elif name.startswith('Furniture state_'):
            input_data[name] = [1 if name == f'Furniture state_{furniture_state}' else 0]
        elif name.startswith('City_'):
            input_data[name] = [1 if name == f'City_{city}' else 0]
        else:
            input_data[name] = [0]
    
    # Create dataframe with correct column order
    X_input = pd.DataFrame(input_data)[feature_names]
    
    # Predict
    prediction = model.predict(X_input)[0]
    probabilities = model.predict_proba(X_input)[0]
    
    return prediction, dict(zip(model.classes_, probabilities))


# Example prediction
print('=== EXAMPLE PREDICTION ===\n')
pred, probs = predict_price_segment(
    area=100,
    frontage=5,
    access_road=10,
    floors=3,
    bedrooms=4,
    bathrooms=3,
    legal_status='Have certificate',
    furniture_state='Full',
    city='H\u00e0 N\u1ed9i'
)

print(f'Predicted Price Segment: {pred}')
print(f'Probabilities:')
for segment, prob in probs.items():
    print(f'  {segment}: {prob:.2%}')

## Classification Summary

In [ ]:
print('='*60)
print('CLASSIFICATION MODELS - SUMMARY')
print('='*60)
print(f'\nDataset size: {len(df_clean):,} records')
print(f'Features: {len(feature_names)} (after encoding)')
print(f'Train/Test split: 80%/20%')
print(f'\nPrice Segments:')
print(f'  Low: Price < {low_threshold:.2f} Billion VND')
print(f'  Medium: {low_threshold:.2f} <= Price < {high_threshold:.2f} Billion VND')
print(f'  High: Price >= {high_threshold:.2f} Billion VND')
print(f'\nDecision Tree Performance:')
print(f'  Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}')
print(f'  F1 Score: {f1_score(y_test, y_pred_dt, average="macro"):.4f}')
print(f'\nRandom Forest Performance:')
print(f'  Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}')
print(f'  F1 Score: {f1_score(y_test, y_pred_rf, average="macro"):.4f}')
print(f'\nBest Model: Random Forest')
print('='*60)